# Evaluate a frozen forward policy

Draw fresh terminal states from saved policies with `epsilon=0` and temperature 1.
No training is resumed and no replay-buffer samples are substituted.

**Inputs:** run configurations and policy checkpoints. **Outputs:** sampled
states/rewards, per-checkpoint metrics, cohort summaries, projection figures, and
input hashes. See [controls](../../README.md#controls) and the
[implementation](analysis_utils.py).

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "code/train.py").is_file() and (p / "requirements.txt").is_file())
for folder in (ROOT / "code", ROOT / "code/notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from analysis_utils import AnalysisSession, CheckpointAnalysis

## 1. Choose runs and sampling settings

`None` preserves `GF_*` overrides. Defaults are 256 samples, sampling seed 42,
the final checkpoint, and CUDA when available. Hypergrid paper tables use
100,000 samples. For `grid-long`, checkpoint defaults are 2k/4k/6k/8k/10k.
Training seeds are never replaced by repeated evaluation seeds.

In [ ]:
# None keeps the GF_* environment setting, or its documented default.
PROFILE = None       # e.g. "smoke", "main", "ablation", "grid-long", "rna-long"
RUN_ROOT = None      # e.g. "outputs/quickstart"; a suite or single run
FIGURE_ROOT = None   # e.g. "figures/my-analysis"; inside this repository
STRICT = None        # True rejects incomplete/incompatible analyses

session = AnalysisSession.from_environment(
    "checkpoint_evaluation.ipynb", root=ROOT, profile=PROFILE, run_root=RUN_ROOT,
    figure_root=FIGURE_ROOT, strict=STRICT, warn_h16_target=True,
)

In [ ]:
SAMPLES = None       # e.g. 100000; GF_EVAL_SAMPLES
SAMPLING_SEED = None # default 42; GF_EVAL_SEED
CHECKPOINTS = None   # e.g. [2000, 4000, 6000, 8000, 10000] or ["final"]
DEVICE = None        # "cpu" or "cuda"; GF_DEVICE

analysis = CheckpointAnalysis(
    session, samples=SAMPLES, seed=SAMPLING_SEED,
    checkpoints=CHECKPOINTS, device=DEVICE,
)

## 2. Sample and evaluate

Hypergrid reports empirical total L1 against the reward-normalized target.
RNA `test_modes` and final-policy sEH metrics are auxiliary diagnostics: the
paper's RNA/sEH counts accumulate during training. Training-discovery values are
read separately when available. Source checks distinguish current float64 rewards
from historical Hypergrid implementations.

In [ ]:
evaluation = analysis.evaluate()
display(evaluation)

## 3. Compare checkpoint projections

Each composite preserves checkpoint order and plots coordinate pairs 1/2 and 3/4
with shared scales across time. Empty panels stay empty. These projections do not
establish coverage of the full joint state space.

In [ ]:
analysis.plot_snapshots()

## 4. Summarize and validate

Cohorts retain the training configuration and source identity. Summaries use
sample standard deviation (`ddof=1`) across training seeds. The manifest records
the sampling budget, seed, temperature, and checkpoint selection.

In [ ]:
summary = analysis.summarize()
display(summary)
manifest = analysis.finish()